In [ ]:
!pip install tenseal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 46.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#!/usr/bin/env python3
"""
Complete FHE Dataset Generation System with Comprehensive Feature Logging
Generates noise-proxy-based dataset for ML-guided bootstrapping research

Features logged per operation:
- Noise proxy (decryption error)
- Operation type, timing, position
- Circuit context (poly_modulus_degree, scale, complexity)
- Input/output magnitudes
- All metadata needed for advanced feature engineering
"""

import tenseal as ts
import pandas as pd
import numpy as np
import json
import time
import os
import logging
from datetime import datetime
from typing import List, Dict, Any, Tuple, Optional
import uuid
from pathlib import Path

L=300
# ============================================================================
# LOGGING SETUP
# ============================================================================

def setup_logging(log_dir: str = "logs") -> logging.Logger:
    """Setup comprehensive logging system"""
    os.makedirs(log_dir, exist_ok=True)
    logger = logging.getLogger('FHE_Dataset_Generator')
    logger.setLevel(logging.INFO)

    # Clear existing handlers
    for h in logger.handlers[:]:
        logger.removeHandler(h)

    # File handler
    log_file = os.path.join(log_dir, f"generation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
    fh = logging.FileHandler(log_file)
    fh.setLevel(logging.INFO)

    # Console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)

    # Formatter
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)

    logger.addHandler(fh)
    logger.addHandler(ch)

    return logger

# ============================================================================
# CONFIGURATION
# ============================================================================
class DatasetConfig:
    """Central configuration for dataset generation"""

    def __init__(self):
        # CKKS parameter sets by complexity
        self.complexity_levels = {
            'simple': {
                'poly_modulus_degree': 4096,
                'coeff_mod_bit_sizes': [30, 20, 30],
                'global_scale': 2**20,
                'max_mult_depth': 2,
                'description': 'Basic circuits, 1-2 multiplications'
            },
            'moderate': {
                'poly_modulus_degree': 8192,
                'coeff_mod_bit_sizes': [30, 20, 20, 30],
                'global_scale': 2**20,
                'max_mult_depth': 3,
                'description': 'Medium circuits, 2-3 multiplications'
            }
        }

        # Circuit counts to generate
        self.circuit_counts = {
            'addition_chains': 5*L,
            'multiplication_chains': 4*L,
            'polynomial_eval': 5*L,
            'neural_layers': 3*L,
            'mixed_ops': 4*L
        }

        # Parameter ranges for each circuit type
        self.param_ranges = {
            'addition_lengths': {
                'simple': [3, 5, 7],
                'moderate': [5, 8, 10]
            },
            'mult_lengths': {
                'simple': [2],
                'moderate': [2, 3]
            },
            'poly_degrees': {
                'simple': [2],
                'moderate': [2, 3]
            },
            'neural_architectures': {
                'simple': [(2, 2), (3, 2)],
                'moderate': [(3, 3), (4, 3)]
            }
        }

        # Value ranges for circuit inputs
        self.value_ranges = {
            'small': (-0.5, 0.5),
            'medium': (-1.0, 1.0)
        }

# ============================================================================
# ADVANCED NOISE LOGGER
# ============================================================================
class ComprehensiveNoiseLogger:
    """
    Logs all operation details with noise proxies and metadata
    for comprehensive feature engineering
    """

    def __init__(self, circuit_id: str, circuit_type: str, complexity: str,
                 context_params: Dict, logger: logging.Logger):
        self.circuit_id = circuit_id
        self.circuit_type = circuit_type  # NEW: track circuit type
        self.complexity = complexity
        self.context_params = context_params
        self.logger = logger

        self.logs = []
        self.op_counter = 0
        self.start_time = time.time()
        self.bootstrap_count = 0
        self.last_bootstrap_time = 0.0
        self.last_bootstrap_op = 0

        # Cumulative tracking
        self.cumulative_mult_depth = 0
        self.cumulative_error = 0.0
        self.cumulative_op_time = 0.0

        # Store plaintext values for magnitude tracking
        self.plaintext_values = []

    def log_operation(self,
                     op_type: str,
                     inputs: List[ts.CKKSVector],
                     output: ts.CKKSVector,
                     op_time: float,
                     expected_output: Optional[float] = None,
                     input_plaintext_values: Optional[List[float]] = None,
                     metadata: Optional[Dict] = None) -> ts.CKKSVector:
        """
        Comprehensive operation logging with all features

        Args:
            op_type: Operation type (add, multiply, square, scalar_mult)
            inputs: Input ciphertext vectors
            output: Output ciphertext vector
            op_time: Operation execution time
            expected_output: Expected plaintext result for noise estimation
            input_plaintext_values: Original plaintext input values
            metadata: Additional operation metadata
        """

        self.op_counter += 1
        current_time = time.time() - self.start_time
        self.cumulative_op_time += op_time

        # Track multiplicative depth
        if op_type in ['multiply', 'square']:
            self.cumulative_mult_depth += 1

        # Estimate noise via decryption error
        noise_proxy = 0.0
        decryption_successful = True
        actual_output = None

        if expected_output is not None:
            try:
                decrypted = output.decrypt()
                actual_output = decrypted[0] if isinstance(decrypted, list) else decrypted

                # Calculate relative error as noise proxy
                if abs(expected_output) > 1e-10:
                    noise_proxy = abs((actual_output - expected_output) / expected_output)
                else:
                    noise_proxy = abs(actual_output - expected_output)

                self.cumulative_error += noise_proxy

            except Exception as e:
                decryption_successful = False
                noise_proxy = float('inf')
                self.logger.warning(f"⚠️  Decryption failed at op {self.op_counter}: {e}")

        # Determine if bootstrap needed (threshold at 1% error)
        bootstrap_needed = noise_proxy > 0.01 or not decryption_successful

        # Calculate input/output magnitudes
        op_input_magnitude = 0.0
        if input_plaintext_values:
            op_input_magnitude = float(np.mean(np.abs(input_plaintext_values)))

        op_output_magnitude = 0.0
        if actual_output is not None:
            op_output_magnitude = float(abs(actual_output))

        # Create comprehensive log entry with ALL features
        log_entry = {
            # ===== CIRCUIT IDENTIFICATION =====
            'circuit_id': self.circuit_id,
            'circuit_type': self.circuit_type,
            'complexity_level': self.complexity,
            'operation_id': self.op_counter,

            # ===== TIMING =====
            'timestamp': datetime.now().isoformat(),
            'relative_time': current_time,
            'operation_time': op_time,
            'cumulative_op_time': self.cumulative_op_time,

            # ===== OPERATION DETAILS =====
            'operation_type': op_type,
            'num_inputs': len(inputs),
            'op_input_magnitude': op_input_magnitude,
            'op_output_magnitude': op_output_magnitude,

            # ===== NOISE METRICS (KEY TARGETS) =====
            'noise_proxy': noise_proxy,
            'cumulative_noise': self.cumulative_error,
            'decryption_error_mean': noise_proxy,  # Alias for clarity
            'decryption_error_max': noise_proxy,   # Single value; extend if vector
            'bootstrap_needed': int(bootstrap_needed),
            'decryption_successful': int(decryption_successful),

            # ===== CIRCUIT CONTEXT =====
            'multiplicative_depth': self.cumulative_mult_depth,
            'position_in_circuit': self.op_counter,
            'operations_since_bootstrap': self.op_counter - self.last_bootstrap_op,
            'time_since_last_bootstrap': current_time - self.last_bootstrap_time,
            'bootstrap_count_so_far': self.bootstrap_count,

            # ===== OPERATION TYPE FLAGS =====
            'is_multiplication': int(op_type == 'multiply'),
            'is_addition': int(op_type == 'add'),
            'is_square': int(op_type == 'square'),
            'is_scalar_mult': int(op_type == 'scalar_mult'),

            # ===== CKKS PARAMETERS (repeated for all ops) =====
            'poly_modulus_degree': self.context_params['poly_modulus_degree'],
            'global_scale': self.context_params['global_scale'],
            'global_scale_log2': int(np.log2(self.context_params['global_scale'])),
            'max_mult_depth_context': self.context_params['max_mult_depth'],

            # ===== EXPECTED VS ACTUAL =====
            'expected_output': expected_output,
            'actual_output': actual_output,

            # ===== METADATA =====
            'metadata': metadata or {}
        }

        # Simulate bootstrap if needed
        if bootstrap_needed:
            self.bootstrap_count += 1
            self.last_bootstrap_op = self.op_counter
            self.last_bootstrap_time = current_time
            self.logger.info(f"🔄 Bootstrap #{self.bootstrap_count} at op {self.op_counter}")

        self.logs.append(log_entry)
        return output

    def get_circuit_summary(self) -> Dict:
        """Get summary statistics for this circuit"""
        if not self.logs:
            return {}

        return {
            'circuit_id': self.circuit_id,
            'circuit_type': self.circuit_type,
            'complexity_level': self.complexity,
            'total_operations': len(self.logs),
            'circuit_length': len(self.logs),
            'total_bootstraps': self.bootstrap_count,
            'final_noise': self.cumulative_error,
            'total_time': self.logs[-1]['relative_time'],
            'successful': self.logs[-1]['decryption_successful']
        }

    def save(self, output_dir: str) -> str:
        """Save logs to files"""
        os.makedirs(output_dir, exist_ok=True)
        base = f"{self.circuit_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

        # Save JSON
        json_path = os.path.join(output_dir, f"{base}.json")
        with open(json_path, 'w') as f:
            json.dump({
                'circuit_summary': self.get_circuit_summary(),
                'operations': self.logs
            }, f, indent=2, default=str)

        # Save CSV
        csv_path = os.path.join(output_dir, f"{base}.csv")
        pd.DataFrame(self.logs).to_csv(csv_path, index=False)

        return base

# ============================================================================
# INSTRUMENTED OPERATIONS
# ============================================================================
class InstrumentedOperations:
    """Wrapper for FHE operations with comprehensive logging"""

    def __init__(self, logger: ComprehensiveNoiseLogger):
        self.logger = logger

    def add(self, a: ts.CKKSVector, b: ts.CKKSVector,
            expected: Optional[float] = None,
            plaintext_vals: Optional[List[float]] = None) -> ts.CKKSVector:
        """Logged addition"""
        t0 = time.time()
        result = a + b
        dt = time.time() - t0
        return self.logger.log_operation('add', [a, b], result, dt, expected, plaintext_vals)

    def multiply(self, a: ts.CKKSVector, b: ts.CKKSVector,
                expected: Optional[float] = None,
                plaintext_vals: Optional[List[float]] = None) -> ts.CKKSVector:
        """Logged multiplication"""
        t0 = time.time()
        result = a * b
        dt = time.time() - t0
        return self.logger.log_operation('multiply', [a, b], result, dt, expected, plaintext_vals)

    def square(self, a: ts.CKKSVector,
              expected: Optional[float] = None,
              plaintext_vals: Optional[List[float]] = None) -> ts.CKKSVector:
        """Logged square"""
        t0 = time.time()
        result = a.square()
        dt = time.time() - t0
        return self.logger.log_operation('square', [a], result, dt, expected, plaintext_vals)

    def scalar_mult(self, a: ts.CKKSVector, scalar: float,
                   expected: Optional[float] = None,
                   plaintext_vals: Optional[List[float]] = None) -> ts.CKKSVector:
        """Logged scalar multiplication"""
        t0 = time.time()
        result = a * scalar
        dt = time.time() - t0
        return self.logger.log_operation('scalar_mult', [a], result, dt, expected,
                                        plaintext_vals, {'scalar': scalar})

# ============================================================================
# CIRCUIT PATTERNS
# ============================================================================
class CircuitPatterns:
    """Collection of FHE circuit generation patterns"""

    @staticmethod
    def addition_chain(context: ts.Context, logger: ComprehensiveNoiseLogger,
                      length: int, value_range: Tuple[float, float]):
        """Addition chain: sum of n values"""
        ops = InstrumentedOperations(logger)

        values = np.random.uniform(value_range[0], value_range[1], length)
        vecs = [ts.ckks_vector(context, [float(v)]) for v in values]

        result = vecs[0]
        running_sum = float(values[0])

        for i in range(1, length):
            running_sum += float(values[i])
            result = ops.add(result, vecs[i], expected=running_sum,
                           plaintext_vals=[values[i-1], values[i]])

        logger.logger.info(f"✅ Addition chain: length={length}, sum={running_sum:.4f}")

    @staticmethod
    def multiplication_chain(context: ts.Context, logger: ComprehensiveNoiseLogger,
                           length: int, value_range: Tuple[float, float]):
        """Multiplication chain"""
        ops = InstrumentedOperations(logger)

        # Constrain values for stability
        small_range = (max(value_range[0], 1.01), min(value_range[1], 1.3))
        values = np.random.uniform(small_range[0], small_range[1], length)
        vecs = [ts.ckks_vector(context, [float(v)]) for v in values]

        result = vecs[0]
        running_prod = float(values[0])

        for i in range(1, length):
            running_prod *= float(values[i])
            result = ops.multiply(result, vecs[i], expected=running_prod,
                                plaintext_vals=[values[i-1], values[i]])

        logger.logger.info(f"✅ Multiplication chain: length={length}, product={running_prod:.4f}")

    @staticmethod
    def polynomial_degree_2(context: ts.Context, logger: ComprehensiveNoiseLogger,
                          value_range: Tuple[float, float]):
        """Polynomial: ax² + bx + c"""
        ops = InstrumentedOperations(logger)

        a, b, c = np.random.uniform(-0.5, 0.5, 3)
        x_val = float(np.random.uniform(value_range[0]/2, value_range[1]/2))

        x = ts.ckks_vector(context, [x_val])

        # x²
        x_squared = ops.square(x, expected=x_val**2, plaintext_vals=[x_val])

        # ax²
        a_vec = ts.ckks_vector(context, [float(a)])
        ax2 = ops.multiply(a_vec, x_squared, expected=a*x_val**2, plaintext_vals=[a, x_val**2])

        # bx
        b_vec = ts.ckks_vector(context, [float(b)])
        bx = ops.multiply(b_vec, x, expected=b*x_val, plaintext_vals=[b, x_val])

        # ax² + bx
        ax2_bx = ops.add(ax2, bx, expected=a*x_val**2 + b*x_val,
                        plaintext_vals=[a*x_val**2, b*x_val])

        # ax² + bx + c
        c_vec = ts.ckks_vector(context, [float(c)])
        result = ops.add(ax2_bx, c_vec, expected=a*x_val**2 + b*x_val + c,
                        plaintext_vals=[a*x_val**2 + b*x_val, c])

        logger.logger.info(f"✅ Polynomial: result={a*x_val**2 + b*x_val + c:.4f}")

    @staticmethod
    def simple_neural_layer(context: ts.Context, logger: ComprehensiveNoiseLogger,
                          input_size: int, output_size: int, value_range: Tuple[float, float]):
        """Neural layer with activation"""
        ops = InstrumentedOperations(logger)

        inputs = np.random.uniform(value_range[0], value_range[1], input_size)
        weights = np.random.uniform(-0.3, 0.3, (output_size, input_size))
        biases = np.random.uniform(-0.1, 0.1, output_size)

        for i in range(output_size):
            weighted_sum = None
            expected_sum = float(biases[i])

            for j in range(input_size):
                inp = ts.ckks_vector(context, [float(inputs[j])])
                term = ops.scalar_mult(inp, float(weights[i, j]),
                                     expected=inputs[j]*weights[i, j],
                                     plaintext_vals=[inputs[j]])
                expected_sum += float(inputs[j] * weights[i, j])

                if weighted_sum is None:
                    weighted_sum = term
                else:
                    weighted_sum = ops.add(weighted_sum, term, expected=expected_sum,
                                         plaintext_vals=[float(inputs[j]*weights[i, j])])

            # Add bias
            bias_vec = ts.ckks_vector(context, [float(biases[i])])
            linear = ops.add(weighted_sum, bias_vec, expected=expected_sum,
                           plaintext_vals=[expected_sum - biases[i], biases[i]])

            # Activation: 0.5x + 0.1x²
            x_sq = ops.square(linear, expected=expected_sum**2, plaintext_vals=[expected_sum])
            term1 = ops.scalar_mult(linear, 0.5, expected=0.5*expected_sum, plaintext_vals=[expected_sum])
            term2 = ops.scalar_mult(x_sq, 0.1, expected=0.1*expected_sum**2, plaintext_vals=[expected_sum**2])
            activated = ops.add(term1, term2, expected=0.5*expected_sum + 0.1*expected_sum**2,
                              plaintext_vals=[0.5*expected_sum, 0.1*expected_sum**2])

        logger.logger.info(f"✅ Neural layer: {input_size}→{output_size}")

    @staticmethod
    def mixed_circuit(context: ts.Context, logger: ComprehensiveNoiseLogger,
                     value_range: Tuple[float, float]):
        """Mixed operations: (a+b)*c² + d"""
        ops = InstrumentedOperations(logger)

        vals = np.random.uniform(value_range[0], value_range[1], 4)
        vecs = [ts.ckks_vector(context, [float(v)]) for v in vals]

        # a + b
        sum_ab = ops.add(vecs[0], vecs[1], expected=vals[0]+vals[1],
                        plaintext_vals=[vals[0], vals[1]])

        # c²
        c_sq = ops.square(vecs[2], expected=vals[2]**2, plaintext_vals=[vals[2]])

        # (a+b) * c²
        prod = ops.multiply(sum_ab, c_sq, expected=(vals[0]+vals[1])*vals[2]**2,
                          plaintext_vals=[vals[0]+vals[1], vals[2]**2])

        # + d
        result = ops.add(prod, vecs[3], expected=(vals[0]+vals[1])*vals[2]**2 + vals[3],
                        plaintext_vals=[(vals[0]+vals[1])*vals[2]**2, vals[3]])

        logger.logger.info(f"✅ Mixed circuit complete")

# ============================================================================
# MAIN DATASET GENERATOR
# ============================================================================
class DatasetGenerator:
    """Main orchestrator for dataset generation"""

    def __init__(self, config: DatasetConfig, output_dir: str = "/content/drive/My Drive/fhe_dataset"): # Modified output_dir
        self.config = config
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)
        self.logger = setup_logging(str(self.output_dir / "logs"))

        self.all_operation_data = []
        self.circuit_summaries = []

    def create_context(self, complexity: str) -> Tuple[ts.Context, Dict]:
        """Create CKKS context"""
        params = self.config.complexity_levels[complexity]

        context = ts.context(
            ts.SCHEME_TYPE.CKKS,
            poly_modulus_degree=params['poly_modulus_degree'],
            coeff_mod_bit_sizes=params['coeff_mod_bit_sizes']
        )
        context.generate_galois_keys()
        context.global_scale = params['global_scale']

        return context, params

    def generate_batch(self, circuit_type: str, complexity: str, count: int):
        """Generate a batch of circuits"""
        self.logger.info(f"\n{'='*70}")
        self.logger.info(f"🔄 Generating {count} {circuit_type} circuits ({complexity} level)")
        self.logger.info(f"{'='*70}")

        context, params = self.create_context(complexity)
        value_range = self.config.value_ranges['small'] if complexity == 'simple' else self.config.value_ranges['medium']

        for i in range(count):
            circuit_id = f"{circuit_type}_{complexity}_{i:04d}_{uuid.uuid4().hex[:6]}"
            circuit_logger = ComprehensiveNoiseLogger(circuit_id, circuit_type, complexity, params, self.logger)

            try:
                # Generate based on type
                if circuit_type == 'addition_chains':
                    length = np.random.choice(self.config.param_ranges['addition_lengths'][complexity])
                    CircuitPatterns.addition_chain(context, circuit_logger, length, value_range)

                elif circuit_type == 'multiplication_chains':
                    length = np.random.choice(self.config.param_ranges['mult_lengths'][complexity])
                    CircuitPatterns.multiplication_chain(context, circuit_logger, length, value_range)

                elif circuit_type == 'polynomial_eval':
                    CircuitPatterns.polynomial_degree_2(context, circuit_logger, value_range)

                elif circuit_type == 'neural_layers':
                    arch_idx = np.random.choice(len(self.config.param_ranges['neural_architectures'][complexity]))
                    inp, out = self.config.param_ranges['neural_architectures'][complexity][arch_idx]
                    CircuitPatterns.simple_neural_layer(context, circuit_logger, inp, out, value_range)

                elif circuit_type == 'mixed_ops':
                    CircuitPatterns.mixed_circuit(context, circuit_logger, value_range)

                # Save individual circuit
                circuit_logger.save(str(self.output_dir / "raw_circuits"))

                # Aggregate
                for log in circuit_logger.logs:
                    self.all_operation_data.append(log)

                self.circuit_summaries.append(circuit_logger.get_circuit_summary())

            except Exception as e:
                self.logger.error(f"❌ Failed {circuit_id}: {e}")
                import traceback
                traceback.print_exc()
                continue

            if (i + 1) % 10 == 0:
                self.logger.info(f"  Progress: {i+1}/{count} ({(i+1)/count*100:.1f}%)")

    def generate_complete_dataset(self):
        """Generate complete dataset"""
        self.logger.info("\n" + "="*70)
        self.logger.info("🚀 STARTING COMPLETE DATASET GENERATION")
        self.logger.info("="*70 + "\n")

        start_time = time.time()

        for circuit_type, count in self.config.circuit_counts.items():
            for complexity in self.config.complexity_levels.keys():
                self.generate_batch(circuit_type, complexity, count)

        total_time = time.time() - start_time

        self.logger.info(f"\n{'='*70}")
        self.logger.info(f"✅ RAW GENERATION COMPLETE in {total_time/60:.1f} minutes")
        self.logger.info(f"{'='*70}\n")

        self.finalize()

    def finalize(self):
        """Process and save final dataset"""
        self.logger.info("📊 Processing and saving final dataset...")

        final_dir = self.output_dir / "final_dataset"
        final_dir.mkdir(exist_ok=True)

        # Create DataFrame
        df = pd.DataFrame(self.all_operation_data)

        self.logger.info(f"   Total operations logged: {len(df)}")
        self.logger.info(f"   Unique circuits: {df['circuit_id'].nunique()}")

        # Save raw
        df.to_csv(final_dir / "raw_operations.csv", index=False)
        df.to_pickle(final_dir / "raw_operations.pkl")

        # Save circuit summaries
        pd.DataFrame(self.circuit_summaries).to_csv(final_dir / "circuit_summaries.csv", index=False)

        self.logger.info(f"✅ Dataset saved to {final_dir}")
        self.logger.info(f"   Next: Run feature engineering script")

# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    print("\n" + "="*70)
    print("🔐 FHE DATASET GENERATION SYSTEM")
    print("   Complete Noise Logging with Comprehensive Features")
    print("="*70 + "\n")

    config = DatasetConfig()
    # Modified output_dir to save directly to Google Drive
    generator = DatasetGenerator(config, output_dir="/content/drive/My Drive/fhe_dataset")
    generator.generate_complete_dataset()

    print("\n" + "="*70)
    print("✅ DATASET GENERATION COMPLETE!")
    print("="*70)
    print("\nNext steps:")
    print("1. Check fhe_dataset/final_dataset/raw_operations.csv in your Google Drive")
    print("2. Run feature_engineering.py to add all advanced features")
    print("3. Begin ML model training")

if __name__ == "__main__":
    main()


🔐 FHE DATASET GENERATION SYSTEM
   Complete Noise Logging with Comprehensive Features



Streaming output truncated to the last 5000 lines.
2025-11-02 11:59:56,320 - INFO - ✅ Mixed circuit complete
INFO:FHE_Dataset_Generator:✅ Mixed circuit complete
2025-11-02 11:59:56,419 - INFO - 🔄 Bootstrap #1 at op 2
INFO:FHE_Dataset_Generator:🔄 Bootstrap #1 at op 2
2025-11-02 11:59:56,433 - INFO - 🔄 Bootstrap #2 at op 3
INFO:FHE_Dataset_Generator:🔄 Bootstrap #2 at op 3
2025-11-02 11:59:56,439 - INFO - 🔄 Bootstrap #3 at op 4
INFO:FHE_Dataset_Generator:🔄 Bootstrap #3 at op 4
2025-11-02 11:59:56,444 - INFO - ✅ Mixed circuit complete
INFO:FHE_Dataset_Generator:✅ Mixed circuit complete
2025-11-02 11:59:56,541 - INFO - 🔄 Bootstrap #1 at op 2
INFO:FHE_Dataset_Generator:🔄 Bootstrap #1 at op 2
2025-11-02 11:59:56,557 - INFO - 🔄 Bootstrap #2 at op 3
INFO:FHE_Dataset_Generator:🔄 Bootstrap #2 at op 3
2025-11-02 11:59:56,562 - INFO - 🔄 Bootstrap #3 at op 4
INFO:FHE_Dataset_Generator:🔄 Bootstrap #3 at op 4
2025-11-02 11:59:56,566 - INFO - ✅ Mixed circuit complete
INFO:FHE_Dataset_Generator:✅ Mixed 


✅ DATASET GENERATION COMPLETE!

Next steps:
1. Check fhe_dataset/final_dataset/raw_operations.csv in your Google Drive
2. Run feature_engineering.py to add all advanced features
3. Begin ML model training


In [ ]:
#!/usr/bin/env python3
"""
Comprehensive Feature Engineering for FHE Bootstrapping Dataset

Adds ALL requested features:
- Circuit-level context
- Rolling/window features
- Positional/sequence features
- Derived interaction features
- Diagnostic flags
"""

import pandas as pd
import numpy as np
import json
from pathlib import Path

def comprehensive_feature_engineering(df: pd.DataFrame, window_sizes=[3, 5, 10]) -> pd.DataFrame:
    """
    Add all advanced features to the dataset

    Features added:
    A. Circuit-level context
    B. Operation-level extended
    C. Noise proxy & labels
    D. Rolling/window features
    E. Positional/sequence
    F. Derived interactions
    G. Diagnostic flags
    """

    print("🔧 Starting comprehensive feature engineering...")
    print(f"   Input: {len(df)} operations")

    # Sort by circuit and operation
    df = df.sort_values(['circuit_id', 'operation_id']).reset_index(drop=True)

    # ========== A. CIRCUIT-LEVEL CONTEXT ==========
    print("   Adding circuit-level features...")

    # Impute missing poly_modulus_degree
    df['context_poly_modulus_missing'] = df['poly_modulus_degree'].isnull().astype(int)
    df['poly_modulus_degree'] = df.groupby('complexity_level')['poly_modulus_degree'].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x.mean()))

    # Ensure global_scale_log2 present
    if 'global_scale_log2' not in df.columns:
        df['global_scale_log2'] = df['global_scale'].apply(lambda x: int(np.log2(x)) if pd.notnull(x) and x > 0 else np.nan)
    df['global_scale_log2'] = df.groupby('complexity_level')['global_scale_log2'].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x.mean()))

    # Circuit length
    df['circuit_length'] = df.groupby('circuit_id')['operation_id'].transform('max')

    # Circuit type (already logged)

    # Start noise proxy
    df['start_noise_proxy'] = df.groupby('circuit_id')['noise_proxy'].transform('first')

    # ========== B. OPERATION-LEVEL EXTENDED ==========
    print("   Adding operation-level features...")

    # Already have: is_multiplication, is_addition, is_square, is_scalar_mult
    # Ensure all present
    for op in ['multiplication', 'addition', 'square', 'scalar_mult']:
        col = f'is_{op}'
        if col not in df.columns:
            df[col] = (df['operation_type'] == op.replace('_', '')).astype(int)

    # Operation magnitudes (already logged as op_input_magnitude, op_output_magnitude)
    # Fill any missing with 0
    for col in ['op_input_magnitude', 'op_output_magnitude']:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # ========== C. NOISE PROXY & LABELS ==========
    print("   Adding noise/label features...")

    # Delta features
    df['delta_decryption_error'] = df.groupby('circuit_id')['decryption_error_mean'].diff().fillna(0)
    df['delta_cumulative_noise'] = df.groupby('circuit_id')['cumulative_noise'].diff().fillna(0)

    # Time to failure (lookahead - simplified: assume failure when bootstrap_needed=1)
    df['time_to_bootstrap'] = 0
    for circuit in df['circuit_id'].unique():
        mask = df['circuit_id'] == circuit
        bootstrap_ops = df[mask & (df['bootstrap_needed'] == 1)]['operation_id'].values

        for idx in df[mask].index:
            op_id = df.loc[idx, 'operation_id']
            future_bootstraps = bootstrap_ops[bootstrap_ops > op_id]
            if len(future_bootstraps) > 0:
                df.loc[idx, 'time_to_bootstrap'] = future_bootstraps[0] - op_id
            else:
                df.loc[idx, 'time_to_bootstrap'] = df.loc[idx, 'circuit_length'] - op_id

    # ========== D. ROLLING/WINDOW FEATURES ==========
    print("   Adding rolling window features...")

    for k in window_sizes:
        print(f"      Window size {k}...")

        # Rolling counts/proportions
        df[f'rolling_count_mult_{k}'] = df.groupby('circuit_id')['is_multiplication'].rolling(
            k, min_periods=1).sum().reset_index(0, drop=True)
        df[f'rolling_prop_mult_{k}'] = df[f'rolling_count_mult_{k}'] / k

        # Rolling operation time stats
        df[f'rolling_avg_op_time_{k}'] = df.groupby('circuit_id')['operation_time'].rolling(
            k, min_periods=1).mean().reset_index(0, drop=True)
        df[f'rolling_std_op_time_{k}'] = df.groupby('circuit_id')['operation_time'].rolling(
            k, min_periods=1).std().reset_index(0, drop=True).fillna(0)

        # Rolling noise/error stats
        df[f'rolling_avg_decryption_error_{k}'] = df.groupby('circuit_id')['decryption_error_mean'].rolling(
            k, min_periods=1).mean().reset_index(0, drop=True)
        df[f'rolling_max_decryption_error_{k}'] = df.groupby('circuit_id')['decryption_error_mean'].rolling(
            k, min_periods=1).max().reset_index(0, drop=True)
        df[f'rolling_cumulative_noise_{k}'] = df.groupby('circuit_id')['delta_cumulative_noise'].rolling(
            k, min_periods=1).sum().reset_index(0, drop=True)

    # ========== E. POSITIONAL/SEQUENCE FEATURES ==========
    print("   Adding positional/sequence features...")

    # Normalized position
    df['position_in_circuit_norm'] = df['operation_id'] / df['circuit_length']
    df['op_index_from_end'] = df['circuit_length'] - df['operation_id']

    # Previous operation types
    df['prev_op_type'] = df.groupby('circuit_id')['operation_type'].shift(1).fillna('START')
    df['prev2_op_type'] = df.groupby('circuit_id')['operation_type'].shift(2).fillna('START')

    # Encode previous op types
    df['prev_op_type_encoded'] = pd.Categorical(df['prev_op_type']).codes
    df['prev2_op_type_encoded'] = pd.Categorical(df['prev2_op_type']).codes

    # Operations since last bootstrap (already present)
    # Time since last bootstrap (already present)

    # ========== F. DERIVED/INTERACTION FEATURES ==========
    print("   Adding derived interaction features...")

    # Multiplicative depth interactions
    df['mult_depth_times_op_time'] = df['multiplicative_depth'] * df['operation_time']

    # Normalize mult depth by context max
    max_depth_per_complexity = df.groupby('complexity_level')['multiplicative_depth'].transform('max')
    df['mult_depth_ratio'] = df['multiplicative_depth'] / max_depth_per_complexity.replace(0, 1)

    # Cumulative ops normalized
    df['cumulative_operations'] = df.groupby('circuit_id').cumcount() + 1
    df['cum_ops_normalized'] = df['cumulative_operations'] / df['circuit_length']

    # ========== G. DIAGNOSTIC/META FLAGS ==========
    print("   Adding diagnostic flags...")

    # Operation time outliers
    op_time_mean = df.groupby('circuit_id')['operation_time'].transform('mean')
    op_time_std = df.groupby('circuit_id')['operation_time'].transform('std')
    df['operation_time_outlier'] = (np.abs(df['operation_time'] - op_time_mean) > 3 * op_time_std).fillna(False).astype(int)

    # Is terminal/final operation
    df['is_terminal_op'] = (df['operation_id'] == df['circuit_length']).astype(int)
    df['is_final_op'] = df['is_terminal_op']  # Alias

    # ========== ENCODE CATEGORICALS ==========
    print("   Encoding categorical features...")

    df['operation_type_encoded'] = pd.Categorical(df['operation_type']).codes
    df['complexity_encoded'] = pd.Categorical(df['complexity_level']).codes
    df['circuit_type_encoded'] = pd.Categorical(df['circuit_type']).codes

    # ========== FILL REMAINING NAs ==========
    print("   Filling missing values...")

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(0)

    print(f"✅ Feature engineering complete!")
    print(f"   Output: {len(df)} operations with {len(df.columns)} features")

    return df

def generate_summary_report(df: pd.DataFrame, output_path: Path):
    """Generate summary statistics report"""

    summary = {
        'total_operations': len(df),
        'unique_circuits': df['circuit_id'].nunique(),
        'total_features': len(df.columns),
        'complexity_distribution': df['complexity_level'].value_counts().to_dict(),
        'circuit_type_distribution': df['circuit_type'].value_counts().to_dict(),
        'operation_type_distribution': df['operation_type'].value_counts().to_dict(),
        'bootstrap_needed_rate': float(df['bootstrap_needed'].mean()),
        'decryption_success_rate': float(df['decryption_successful'].mean()),
        'noise_statistics': {
            'mean': float(df['noise_proxy'].mean()),
            'std': float(df['noise_proxy'].std()),
            'min': float(df['noise_proxy'].min()),
            'max': float(df['noise_proxy'].max())
        }
    }

    with open(output_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"\n📊 Summary Statistics:")
    for key, value in summary.items():
        if not isinstance(value, dict):
            print(f"   {key}: {value}")

def main():
    print("\n" + "="*70)
    print("🔧 FHE DATASET FEATURE ENGINEERING")
    print("="*70 + "\n")

    # Load raw dataset
    input_path = Path("/content/drive/MyDrive/fhe_dataset/final_dataset/raw_operations.csv")
    output_dir = Path("/content/drive/MyDrive/fhe_dataset/final_dataset")

    if not input_path.exists():
        print(f"❌ Error: {input_path} not found!")
        print("   Please run dataset_generation_complete.py first")
        return

    print(f"📂 Loading dataset from {input_path}...")
    df = pd.read_csv(input_path)

    # Apply feature engineering
    df_engineered = comprehensive_feature_engineering(df)

    # Save engineered dataset
    output_path = output_dir / "ml_training_data.csv"
    df_engineered.to_csv(output_path, index=False)
    print(f"\n💾 Saved to {output_path}")

    # Save as pickle for faster loading
    df_engineered.to_pickle(output_dir / "ml_training_data.pkl")
    print(f"💾 Saved to {output_dir / 'ml_training_data.pkl'}")

    # Generate summary
    generate_summary_report(df_engineered, output_dir / "feature_summary.json")

    # Save feature list
    with open(output_dir / "feature_names.txt", 'w') as f:
        for col in sorted(df_engineered.columns):
            f.write(f"{col}\n")

    print(f"\n{'='*70}")
    print("✅ FEATURE ENGINEERING COMPLETE!")
    print("="*70)
    print("\nNext steps:")
    print("1. Review ml_training_data.csv")
    print("2. Check feature_summary.json for statistics")
    print("3. Begin ML model training with supervised_model_training.py")

if __name__ == "__main__":
    main()



🔧 FHE DATASET FEATURE ENGINEERING

📂 Loading dataset from /content/drive/MyDrive/fhe_dataset/final_dataset/raw_operations.csv...
🔧 Starting comprehensive feature engineering...
   Input: 31184 operations
   Adding circuit-level features...
   Adding operation-level features...
   Adding noise/label features...
   Adding rolling window features...
      Window size 3...
      Window size 5...
      Window size 10...
   Adding positional/sequence features...
   Adding derived interaction features...
   Adding diagnostic flags...
   Encoding categorical features...
   Filling missing values...
✅ Feature engineering complete!
   Output: 31184 operations with 77 features

💾 Saved to /content/drive/MyDrive/fhe_dataset/final_dataset/ml_training_data.csv
💾 Saved to /content/drive/MyDrive/fhe_dataset/final_dataset/ml_training_data.pkl

📊 Summary Statistics:
   total_operations: 31184
   unique_circuits: 8100
   total_features: 77
   bootstrap_needed_rate: 0.4993265777321703
   decryption_succe

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/fhe_dataset/final_dataset/ml_training_data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31184 entries, 0 to 31183
Data columns (total 77 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   circuit_id                       31184 non-null  object 
 1   circuit_type                     31184 non-null  object 
 2   complexity_level                 31184 non-null  object 
 3   operation_id                     31184 non-null  int64  
 4   timestamp                        31184 non-null  object 
 5   relative_time                    31184 non-null  float64
 6   operation_time                   31184 non-null  float64
 7   cumulative_op_time               31184 non-null  float64
 8   operation_type                   31184 non-null  object 
 9   num_inputs                       31184 non-null  int64  
 10  op_input_magnitude               31184 non-null  float64
 11  op_output_magnitude              31184 non-null  float64
 12  noise_proxy       

In [ ]:
df.head()

,circuit_id,circuit_type,complexity_level,operation_id,timestamp,relative_time,operation_time,cumulative_op_time,operation_type,num_inputs,...,mult_depth_times_op_time,mult_depth_ratio,cumulative_operations,cum_ops_normalized,operation_time_outlier,is_terminal_op,is_final_op,operation_type_encoded,complexity_encoded,circuit_type_encoded
0,addition_chains_moderate_0000_0ba7f0,addition_chains,moderate,1,2025-11-01T16:04:07.733691,0.207071,0.002592,0.002592,add,2,...,0.0,0.0,1,0.142857,0,0,0,0,0,0
1,addition_chains_moderate_0000_0ba7f0,addition_chains,moderate,2,2025-11-01T16:04:07.756024,0.226317,0.000514,0.003106,add,2,...,0.0,0.0,2,0.285714,0,0,0,0,0,0
2,addition_chains_moderate_0000_0ba7f0,addition_chains,moderate,3,2025-11-01T16:04:07.763380,0.237279,0.000280,0.003386,add,2,...,0.0,0.0,3,0.428571,0,0,0,0,0,0
3,addition_chains_moderate_0000_0ba7f0,addition_chains,moderate,4,2025-11-01T16:04:07.770844,0.244648,0.000293,0.003679,add,2,...,0.0,0.0,4,0.571429,0,0,0,0,0,0
4,addition_chains_moderate_0000_0ba7f0,addition_chains,moderate,5,2025-11-01T16:04:07.779536,0.252108,0.000286,0.003965,add,2,...,0.0,0.0,5,0.714286,0,0,0,0,0,0


In [ ]:
# Copy the entire directory to Google Drive
# !cp -r /content/fhe_dataset "/content/drive/MyDrive/"